In [1]:
from IPython.display import display
import sympy as sp
from sympy import Function, symbols, cos, sin, dsolve, Eq, Symbol

t = sp.symbols('t')
x = Function('x')(t) # ty: ignore
y = Function('y')(t) # ty: ignore
vh, vv = symbols('v_h v_v')
x0, y0, theta0 = symbols('x0 y0 theta0')

def solve_with(omega):
    theta = omega * t + theta0
    M = sp.Matrix([
        [cos(theta), -sin(theta)], 
        [sin(theta), cos(theta)]
    ])
    p = sp.Matrix([x, y])
    v = sp.Matrix([vh, vv])

    lhs = sp.diff(p, t)
    rhs = M.multiply(v)

    sol = separate(dsolve(
        [Eq(lhs[i], rhs[i]) for i in range(len(rhs))], 
        ics={x.subs(t, 0): x0, y.subs(t, 0): y0}
    ))
    return Eq(lhs, rhs), sol

def separate(expr: list[sp.Eq]) -> sp.Eq:
    lhs = [e.lhs for e in expr]
    rhs = [e.rhs for e in expr]
    return Eq(sp.Matrix(lhs), sp.Matrix(rhs))

eq_zero, sol_zero = solve_with(Symbol('omega', zero=True))
eq_nonzero, sol_nonzero = solve_with(Symbol('omega', zero=False))

display("Differential equation", eq_zero)
display("Zero omega solution", sol_zero)
display("Non-zero omega solution", sol_nonzero)

'Differential equation'

Eq(Matrix([
[Derivative(x(t), t)],
[Derivative(y(t), t)]]), Matrix([
[v_h*cos(omega*t + theta0) - v_v*sin(omega*t + theta0)],
[v_h*sin(omega*t + theta0) + v_v*cos(omega*t + theta0)]]))

'Zero omega solution'

Eq(Matrix([
[x(t)],
[y(t)]]), Matrix([
[t*v_h*cos(omega*t + theta0) - t*v_v*sin(omega*t + theta0) + x0],
[t*v_h*sin(omega*t + theta0) + t*v_v*cos(omega*t + theta0) + y0]]))

'Non-zero omega solution'

Eq(Matrix([
[x(t)],
[y(t)]]), Matrix([
[x0 + v_h*sin(omega*t + theta0)/omega + v_v*cos(omega*t + theta0)/omega - (v_h*sin(theta0) + v_v*cos(theta0))/omega],
[y0 - v_h*cos(omega*t + theta0)/omega + v_v*sin(omega*t + theta0)/omega + (v_h*cos(theta0) - v_v*sin(theta0))/omega]]))

In [2]:
omega = Symbol('omega', zero=False)
subbed = sol_nonzero.expand().subs({
    vh / omega: Symbol('r_h'),
    vv / omega: Symbol('r_v')
})
subbed

Eq(Matrix([
[x(t)],
[y(t)]]), Matrix([
[-r_h*sin(theta0) + r_h*sin(omega*t + theta0) - r_v*cos(theta0) + r_v*cos(omega*t + theta0) + x0],
[ r_h*cos(theta0) - r_h*cos(omega*t + theta0) - r_v*sin(theta0) + r_v*sin(omega*t + theta0) + y0]]))

In [3]:
from sympy import cse


replacements, reduced = cse(subbed)

display(*[Eq(a, b) for a, b in replacements])
display(*reduced)

Eq(x1, sin(theta0))

Eq(x2, cos(theta0))

Eq(x3, omega*t + theta0)

Eq(x4, sin(x3))

Eq(x5, cos(x3))

Eq(Matrix([
[x(t)],
[y(t)]]), Matrix([
[-r_h*x1 + r_h*x4 - r_v*x2 + r_v*x5 + x0],
[ r_h*x2 - r_h*x5 - r_v*x1 + r_v*x4 + y0]]))